#04 - Time Travel Demo

Simula um incidente real: exclusão indevida de dados em silver.orders,

seguida de recuperação via RESTORE TABLE.

## 1. Estado antes do incidente

In [0]:
SELECT COUNT(*) as total_antes FROM olist_project.silver.orders

In [0]:
DESCRIBE HISTORY olist_project.silver.orders

## 2. [INCIDENTE] Exclusão indevida

Simulação: um DELETE sem filtro adequado, apagando todos os pedidos

do estado de SP por engano (deveria ter sido um filtro mais específico).

In [0]:
DELETE FROM olist_project.silver.orders

## 3. Confirmação do estrago

In [0]:
SELECT COUNT(*) as total_depois_do_incidente FROM olist_project.silver.orders

## 4. Identificação da versão a restaurar

In [0]:
DESCRIBE HISTORY olist_project.silver.orders

##5. Recuperação via RESTORE TABLE

In [0]:
RESTORE TABLE olist_project.silver.orders TO VERSION AS OF 14

**Atenção: Ao tentar restaura a VERSÃO 1**
Recebemos o aviso:
```
[DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION] Cannot time travel beyond delta.deletedFileRetentionDuration (168 HOURS) set on the table. SQLSTATE: 0AKDC
```
Na Issue #23, aplicamos VACCUM  e o VACUUM apagou fisicamente os arquivos de dados que não eram mais referenciados por nenhuma versão dentro da janela de 7 dias. Só que a versão "1" que estamos tentando restaurar (do MERGE INTO da Sprint 2, há mais de uma semana) provavelmente já teve seus arquivos removidos por aquele VACUUM, mesmo a entrada no histórico (DESCRIBE HISTORY) ainda existindo como metadado, os arquivos físicos daquela versão específica **não existem mais em disco**.

É exatamente o "ponto de atenção" que a gente documentou na issue #23:

> depois de rodar VACUUM, você perde a capacidade de fazer Time Travel pra versões mais antigas que os arquivos removidos

Isso não é bug nem erro nosso, é o Delta Lake protegendo a gente de tentar restaurar pra um estado cujos dados não existem mais fisicamente.

**Como resolver**

Você precisa fazer o RESTORE pra uma versão mais recente, uma que tenha sido criada depois do VACUUM, e portanto ainda tenha arquivos físicos preservados.

**Passo 1**: rode DESCRIBE HISTORY de novo e olhe as versões mais recentes (as de hoje, próximas do momento em que você rodou o DELETE do incidente). A versão que você quer é a que veio imediatamente antes do DELETE, não a "versão 1" antiga.

```
DESCRIBE HISTORY olist_project.silver.orders
```
**Passo 2**: identifique visualmente qual número de versão corresponde ao estado antes do DELETE que você acabou de rodar agora (deve ser a penúltima linha, cronologicamente falando — a última é o próprio DELETE).

**Passo 3**: ajuste o RESTORE pra esse número certo.

In [0]:
RESTORE TABLE olist_project.silver.orders TO VERSION AS OF 14

## 6. Confirmação da recuperação

In [0]:
SELECT COUNT(*) as total_recuperado FROM olist_project.silver.orders